# Notebook 06 – Master Dataset Quality Control

This notebook audits the actual 84-row master CSV, preserves all source records, corrects the fracture-location column name, creates target-specific eligibility flags, and exports QC/modeling files.

**Important:** eligibility flags are conservative screening decisions, not proof that records are scientifically comparable. Source-level validation and study-grouped validation are still required.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import files
from IPython.display import display

uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith('.csv') and 'master_experimental_dataset' in name.lower()]
if not csv_files:
    raise FileNotFoundError('Upload master_experimental_dataset.csv (a filename suffix such as (1) is also accepted).')
input_file = csv_files[0]
df = pd.read_csv(input_file)
print('Loaded:', input_file, '| Shape:', df.shape)
display(df.head())

## 1. Structural checks and terminology correction

The source column `fracture_area_reported` contains categorical fracture-location descriptions (for example Al, Cu, Joint), so it is renamed without changing its values.

In [ ]:
if 'fracture_area_reported' in df.columns and 'fracture_location_reported' not in df.columns:
    df = df.rename(columns={'fracture_area_reported': 'fracture_location_reported'})

required = ['master_record_id','source_id','study_id','record_id']
missing_required = [c for c in required if c not in df.columns]
if missing_required:
    raise ValueError(f'Missing required identifier columns: {missing_required}')

df['master_record_id'] = df['master_record_id'].astype('string')
df['source_id'] = df['source_id'].astype('string')
df['study_id'] = df['study_id'].astype('string')
df['record_id'] = df['record_id'].astype('string')

print('Rows, columns:', df.shape)
print('Duplicate master IDs:', int(df['master_record_id'].duplicated().sum()))
print('Duplicate source-record keys:', int(df.duplicated(['source_id','record_id']).sum()))
print('Exact duplicate rows:', int(df.duplicated().sum()))
display(df.groupby('source_id', dropna=False).size().rename('records').reset_index())

## 2. Source-specific quality-control audit

Missing values in columns belonging to another study are expected in this combined dataset. Review missingness within each source rather than treating every blank cell as an error.

In [ ]:
source_col = 'source_id'
source_counts = df.groupby(source_col, dropna=False).size().rename('record_count').reset_index()
missing_by_source = df.groupby(source_col, dropna=False).apply(lambda g: g.isna().sum(), include_groups=False).T
missing_by_source.index.name = 'column'
missing_by_source = missing_by_source.reset_index()

exact_zero_cols = [c for c in ['weld_bead_width_mm','interface_width_mm','penetration_depth_mm'] if c in df.columns]
conventional_mask = df[source_col].eq('conventional_laser_welding_dataset')
if exact_zero_cols:
    conventional_all_zero = conventional_mask & df[exact_zero_cols].eq(0).all(axis=1)
else:
    conventional_all_zero = pd.Series(False, index=df.index)

wobble_mask = df[source_col].eq('alcu_wobbling_welding_dataset')
wobble_width_cols = [c for c in ['bead_width_face_mm','bead_width_interface_mm'] if c in df.columns]
wobble_width_missing = wobble_mask & df[wobble_width_cols].isna().any(axis=1) if wobble_width_cols else pd.Series(False, index=df.index)

lee_mask = df['study_id'].eq('study_lee_2022') if 'study_id' in df.columns else pd.Series(False, index=df.index)
lee_flags = pd.Series(False, index=df.index)
for c in ['min_exceeds_max_flag','average_outside_reported_range_flag']:
    if c in df.columns:
        lee_flags |= df[c].fillna(False).astype(bool)

qc_summary = pd.DataFrame([
    ['total_records', len(df)],
    ['duplicate_master_ids', int(df['master_record_id'].duplicated().sum())],
    ['duplicate_source_record_keys', int(df.duplicated(['source_id','record_id']).sum())],
    ['exact_duplicate_rows', int(df.duplicated().sum())],
    ['conventional_all_zero_output_records', int(conventional_all_zero.sum())],
    ['wobbling_records_with_missing_width', int(wobble_width_missing.sum())],
    ['lee_records_with_force_range_flags', int((lee_mask & lee_flags).sum())],
], columns=['audit_item','count'])

display(source_counts)
display(qc_summary)
display(missing_by_source)

## 3. Target-specific modelling eligibility

Each output gets its own eligibility flag. Conventional all-zero output records are retained in the master table but excluded from default regression subsets until their physical meaning is confirmed. Wobbling width records are eligible only when the relevant width exists. Lee 2022 force records remain excluded by default until the source table is verified; change the switch below only after checking the original publication.

In [ ]:
# Keep False until the Lee 2022 force table has been checked against the original publication.
LEE_FORCE_SOURCE_VERIFIED = False

def numeric_present(col):
    return df[col].notna() & pd.to_numeric(df[col], errors='coerce').notna() if col in df.columns else pd.Series(False, index=df.index)

def add_eligibility(flag_name, target_col, allowed_mask, extra_mask=None):
    eligible = allowed_mask & numeric_present(target_col)
    if extra_mask is not None:
        eligible &= extra_mask
    df[flag_name] = eligible.astype(bool)

nonzero_conventional = ~conventional_all_zero
add_eligibility('eligible_weld_bead_width', 'weld_bead_width_mm', conventional_mask, nonzero_conventional)
add_eligibility('eligible_interface_width', 'interface_width_mm', conventional_mask, nonzero_conventional)
add_eligibility('eligible_penetration_depth_conventional', 'penetration_depth_mm', conventional_mask, nonzero_conventional)

add_eligibility('eligible_bead_width_face_wobble', 'bead_width_face_mm', wobble_mask)
add_eligibility('eligible_bead_width_interface_wobble', 'bead_width_interface_mm', wobble_mask)
add_eligibility('eligible_penetration_depth_wobble', 'penetration_depth_mm', wobble_mask)

shin_mask = df[source_col].eq('shin_2021_metals_11050831')
add_eligibility('eligible_tensile_shear_strength', 'tensile_shear_strength_MPa', shin_mask)
add_eligibility('eligible_electrical_resistance', 'electrical_resistance_microohm', shin_mask)
add_eligibility('eligible_welding_area', 'welding_area_mm2', shin_mask)

lee_force_allowed = lee_mask & (~lee_flags) if LEE_FORCE_SOURCE_VERIFIED else pd.Series(False, index=df.index)
add_eligibility('eligible_force_average_N', 'force_average_N_reported', lee_force_allowed)

eligibility_cols = [c for c in df.columns if c.startswith('eligible_')]
display(df.groupby('source_id')[eligibility_cols].sum().fillna(0).astype(int))

## 4. Export target-specific datasets and audit files

These CSVs are separate by target and source. They are not yet a single pooled training dataset; assess study compatibility and use study-grouped validation before fitting models.

In [ ]:
out = Path('notebook06_outputs')
out.mkdir(exist_ok=True)

# Keep the original broad eligibility column for traceability, but do not use it as the modelling filter.
df.to_csv(out/'master_experimental_dataset_qc.csv', index=False)
source_counts.to_csv(out/'source_record_counts.csv', index=False)
qc_summary.to_csv(out/'master_quality_control_summary.csv', index=False)
missing_by_source.to_csv(out/'missingness_by_source.csv', index=False)

target_flags = {
    'conventional_bead_width': 'eligible_weld_bead_width',
    'conventional_interface_width': 'eligible_interface_width',
    'conventional_penetration_depth': 'eligible_penetration_depth_conventional',
    'wobble_face_bead_width': 'eligible_bead_width_face_wobble',
    'wobble_interface_bead_width': 'eligible_bead_width_interface_wobble',
    'wobble_penetration_depth': 'eligible_penetration_depth_wobble',
    'shin_tensile_shear_strength': 'eligible_tensile_shear_strength',
    'shin_electrical_resistance': 'eligible_electrical_resistance',
    'shin_welding_area': 'eligible_welding_area',
    'lee_force_average': 'eligible_force_average_N',
}
for name, flag in target_flags.items():
    subset = df.loc[df[flag]].copy()
    subset.to_csv(out/f'model_data_{name}.csv', index=False)
    print(f'{name}: {len(subset)} eligible rows')

# One concise data dictionary for the newly created eligibility flags.
dictionary = pd.DataFrame([
    [flag, 'Boolean', 'True only when the corresponding target is present and passes the conservative source-specific screening rule.']
    for flag in eligibility_cols
], columns=['column','data_type','definition'])
dictionary.to_csv(out/'eligibility_flag_dictionary.csv', index=False)

print('\nCreated files:')
for f in sorted(out.glob('*.csv')):
    print('-', f.name)


In [ ]:
# Download all outputs as one ZIP archive.
import shutil
zip_path = shutil.make_archive('notebook06_outputs', 'zip', out)
files.download(zip_path)